# Task 3 — RAG Pipeline & Gradio Chat UI
### TechNova Support · RAG Project


## 1. Imports & Config

In [5]:
import pathlib, random
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr
import pandas as pd

load_dotenv(override=True)

MODEL       = "gpt-4.1-nano"
EMBED_MODEL = "all-MiniLM-L6-v2"
DB_NAME     = "vector_db"
KB_PATH     = pathlib.Path("knowledge-base")
TOP_K       = 5

print(f"Model : {MODEL}  |  Embed : {EMBED_MODEL}")

Model : gpt-4.1-nano  |  Embed : all-MiniLM-L6-v2


## 2. Load Tasks 1 & 2

In [9]:
%run Task1_Ingestion.ipynb
%run Task2_VectorStore.ipynb
print(f"\nVector store ready — {vectorstore._collection.count()} chunks")

Imports OK
Loaded 34 documents from knowledge-base/

  [company             ]  company\about.md
  [company             ]  company\careers.md
  [company             ]  company\contact.md
  [company             ]  company\sustainability.md
  [faqs                ]  faqs\account.md
  [faqs                ]  faqs\payments.md
  [faqs                ]  faqs\returns.md
  [faqs                ]  faqs\shipping.md
  [faqs                ]  faqs\software_updates.md
  [faqs                ]  faqs\trade_in.md
  [faqs                ]  faqs\troubleshooting.md
  [faqs                ]  faqs\warranty.md
  [policies            ]  policies\privacy_policy.md
  [policies            ]  policies\return_policy.md
  [policies            ]  policies\shipping_policy.md
  [policies            ]  policies\terms_of_service.md
  [policies            ]  policies\warranty_policy.md
  [products            ]  products\nova_earbuds.md
  [products            ]  products\nova_laptop_pro.md
  [products            ]  produc

e:\TechNova_Support_Final\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Imports OK
Loaded 34 documents from knowledge-base/

  [company             ]  company\about.md
  [company             ]  company\careers.md
  [company             ]  company\contact.md
  [company             ]  company\sustainability.md
  [faqs                ]  faqs\account.md
  [faqs                ]  faqs\payments.md
  [faqs                ]  faqs\returns.md
  [faqs                ]  faqs\shipping.md
  [faqs                ]  faqs\software_updates.md
  [faqs                ]  faqs\trade_in.md
  [faqs                ]  faqs\troubleshooting.md
  [faqs                ]  faqs\warranty.md
  [policies            ]  policies\privacy_policy.md
  [policies            ]  policies\return_policy.md
  [policies            ]  policies\shipping_policy.md
  [policies            ]  policies\terms_of_service.md
  [policies            ]  policies\warranty_policy.md
  [products            ]  products\nova_earbuds.md
  [products            ]  products\nova_laptop_pro.md
  [products            ]  produc

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Loaded existing vector store (104 chunks). Skipping rebuild.

Final collection size : 104
Sample metadata       : {'source': 'company\\about.md', 'start_index': 0, 'category': 'company'}
Retrieval sanity checks:

  Q: 'What is the return window for NovaCare+ members?'
  filter=policies
  sources=['policies\\return_policy.md', 'policies\\warranty_policy.md', 'policies\\return_policy.md']

  Q: 'How much is the Nova Phone X1?'
  filter=None
  sources=['products\\nova_phone_x1.md', 'products\\nova_phone_x1.md', 'products\\nova_tablet.md']

  Q: 'cracked watch screen ticket'
  filter=None
  sources=['support_tickets\\ticket_004.md', 'support_tickets\\ticket_004.md', 'support_tickets\\ticket_008.md']

  Q: 'purple banana spaceship'
  filter=None
  sources=['products\\nova_speaker.md', 'products\\nova_phone_x1.md', 'products\\nova_speaker.md']

Vector store ready — 104 chunks


## 3. Generation with Source Citations


In [10]:
llm = ChatOpenAI(model=MODEL, temperature=0)

SYSTEM_PROMPT = """You are a professional support assistant for TechNova.
You help TechNova's internal support team answer customer questions accurately.

STRICT RULES:
1. Answer ONLY using the CONTEXT chunks provided — never use outside knowledge.
2. After every factual statement cite its source: (source: path/filename.md)
3. If multiple sources support a fact, cite all of them.
4. If the context does NOT contain the answer, respond with exactly:
   \"I don't have that information in my knowledge base.\"
5. NEVER fabricate or invent a source not listed in the context.
6. Use bullet points for multi-part answers. Keep responses concise.

CONTEXT:
{context}
"""


def build_context(docs: list) -> str:
    """Format retrieved chunks with labeled source headers for the prompt."""
    return "\n\n".join(
        f"[{i}] Extract from {d.metadata.get('source', '?')}:\n{d.page_content}"
        for i, d in enumerate(docs, 1)
    )


def answer_question(question: str, history: list, category: str = "All") -> str:
    """
    Full RAG pipeline:
      1. Retrieve top-k chunks (with optional category filter)
      2. Format chunks as labeled context
      3. Inject conversation history for multi-turn awareness
      4. Call GPT and return cited answer
    """
    cat  = category if category not in (None, "All") else None
    docs = retrieve(question, k=TOP_K, category=cat)
    msgs = [SystemMessage(content=SYSTEM_PROMPT.format(context=build_context(docs)))]

   
    for user_msg, bot_msg in history:
        msgs += [HumanMessage(content=user_msg),
                 SystemMessage(content=str(bot_msg))]

    msgs.append(HumanMessage(content=question))
    return llm.invoke(msgs).content


print('=== Policy question ===')
print(answer_question('What is the return window for NovaCare+ members?', []))

print('\n=== Out-of-scope (should refuse) ===')
print(answer_question('What is the weather in Paris tomorrow?', []))

print('\n=== Product question ===')
print(answer_question('How much does the Nova Phone X1 cost?', []))

=== Policy question ===
The return window for NovaCare+ members is extended to 45 days.

=== Out-of-scope (should refuse) ===
I don't have that information in my knowledge base.

=== Product question ===
- 256GB model: $899
- 512GB model: $1,049 (source: products\nova_phone_x1.md)


## 4. Gradio Chat UI 


In [11]:
CATEGORIES = ["All", "products", "faqs", "policies",
              "support_tickets", "company", "release_notes"]

kb_records = []
for md_file in sorted(KB_PATH.rglob('*.md')):
    rel = md_file.relative_to(KB_PATH)
    kb_records.append({
        'Category': rel.parts[0],
        'Source': str(rel),
        'Size (chars)': len(md_file.read_text(encoding='utf-8'))
    })
kb_df = pd.DataFrame(kb_records)


with gr.Blocks(title='TechNova Support', theme=gr.themes.Soft()) as demo:

    # Header
    gr.Markdown("""
    #  TechNova Support Assistant
    **RAG-Powered** internal support chatbot · Chat · Knowledge Base · Evaluation
    """)

    with gr.Tabs():

        with gr.TabItem('Chat'):
            gr.Markdown(
                'Ask anything about TechNova products, policies, shipping, '
                'returns, or support tickets.'
            )
            cat_dd = gr.Dropdown(
                choices=CATEGORIES, value='All',
                label='Category Filter'
            )
            chatbot   = gr.Chatbot(label='TechNova Support', height=430)
            msg_box   = gr.Textbox(
                placeholder='Ask about products, policies, shipping, warranties...',
                label='Your question', lines=1
            )
            with gr.Row():
                send_btn  = gr.Button('Send ▶', variant='primary')
                clear_btn = gr.Button('Clear')

            gr.Examples(
                examples=[
                    'What is the return window for NovaCare+ members?',
                    'How much does the Nova Phone X1 cost?',
                    'Was the cracked screen in ticket 004 covered under warranty?',
                    'What new workout types were added in NovaWatch OS 4.1?',
                    'Does TechNova ship to Russia?',
                    'What is the weather in Paris tomorrow?',
                ],
                inputs=msg_box
            )

            def respond(message, history, category):
                reply = answer_question(message, history, category=category)
                history.append([message, reply])
                return '', history

            send_btn.click(respond,
                           inputs=[msg_box, chatbot, cat_dd],
                           outputs=[msg_box, chatbot])
            msg_box.submit(respond,
                           inputs=[msg_box, chatbot, cat_dd],
                           outputs=[msg_box, chatbot])
            clear_btn.click(lambda: ('', []), outputs=[msg_box, chatbot])

        
        with gr.TabItem('Knowledge Base'):
            gr.Markdown(
                f'**{len(kb_records)} documents** loaded from `knowledge-base/` '
                f'across **{kb_df["Category"].nunique()} categories**'
            )
            cat_filter = gr.Dropdown(
                choices=['All'] + CATEGORIES[1:],
                value='All', label='Filter by Category'
            )
            kb_table = gr.DataFrame(
                value=kb_df, interactive=False,
                label='Knowledge Base Documents'
            )
            cat_filter.change(
                fn=lambda c: kb_df if c == 'All' else kb_df[kb_df['Category'] == c],
                inputs=cat_filter, outputs=kb_table
            )

       
        with gr.TabItem('Evaluation'):
            gr.Markdown('**Full evaluation harness is in Task4_Evaluation.ipynb**')
            gr.Markdown('> Run Task 4 to score all 29 questions and view results here.')

demo.launch()

Running on local URL:  http://127.0.0.1:7866

To create a public link, set `share=True` in `launch()`.
